## 0. Импорт

In [24]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from tqdm.notebook import tqdm

## 1. Предварительная обработка
1) Прочитайте файл day-of-week-not-scaled.csv. Он похож на файл из предыдущего упражнения, но на этот раз мы не масштабировали непрерывные признаки (мы больше не будем использовать logreg). Не забудьте дополнить таблицу столбцом 'dayofweek' из CSV-файла за предыдущий день.
2) Используя train_test_splitпараметры test_size=0.2, random_state=21получите X_train, y_train, X_test, y_test. Используйте дополнительный параметр stratify.

In [3]:
df = pd.read_csv('../data/day-of-week-not-scaled.csv')
lastDf = pd.read_csv('../data/dayofweek.csv')
df = lastDf[['dayofweek'] + [c for c in df if c != 'dayofweek']]
df.head(3)

,dayofweek,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,...,labname_lab02,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1
0,4,-0.788667,-2.562352,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,4,-0.756764,-2.562352,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,4,-0.724861,-2.562352,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [4]:
X = df[df.drop(columns='dayofweek',axis=1).columns]
y = df['dayofweek']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)

## 2. Поиск по сетке с использованием SVM
1) Попробуйте GridSearchCVразные параметры ядра ( linear, rbf, sigmoid), C ( 0.01, 0.1, 1, 1.5, 5, 10), гамма ( scale, auto), веса класса ( balanced, None) random_state=21и probability=Trueнайдите наилучшую их комбинацию с точки зрения точности.
2) Создайте датафрейм из результатов поиска по сетке и отсортируйте его по возрастанию rank_test_score. Проверьте, есть ли существенная разница между различными комбинациями (иногда более простая модель может дать сопоставимый результат).

In [5]:
svc = SVC(random_state=21, probability=True)

param_grid_svc = {'C': [0.01, 0.1, 1, 1.5, 5, 10],
              'kernel': ['linear', 'rbf', 'sigmoid'],
              'gamma': ['scale', 'auto'],
              'class_weight': [None, 'balanced'],
              }

gridSVC = GridSearchCV(svc, param_grid_svc)
gridSVC.fit(X,y)
print("Лучшие параметры:", gridSVC.best_params_)
print("Лучшая точность:", gridSVC.best_score_)

Лучшие параметры: {'C': 5, 'class_weight': None, 'gamma': 'scale', 'kernel': 'rbf'}
Лучшая точность: 0.5238213965901709


In [6]:
results_df_svc = pd.DataFrame(gridSVC.cv_results_)

columns_to_show = ['rank_test_score', 'mean_test_score', 'std_test_score',
                   'param_C', 'param_kernel', 'param_gamma', 'param_class_weight']

results_sorted_svc = results_df_svc[columns_to_show].sort_values('rank_test_score')

results_sorted_svc.head(15)

,rank_test_score,mean_test_score,std_test_score,param_C,param_kernel,param_gamma,param_class_weight
49,1,0.523821,0.127484,5.0,rbf,scale,None
67,2,0.520859,0.127983,10.0,rbf,scale,balanced
61,3,0.514909,0.118515,10.0,rbf,scale,None
55,4,0.511977,0.131146,5.0,rbf,scale,balanced
37,5,0.498931,0.138515,1.5,rbf,scale,None
43,6,0.492410,0.137716,1.5,rbf,scale,balanced
25,7,0.487075,0.142061,1.0,rbf,scale,None
31,8,0.474616,0.135092,1.0,rbf,scale,balanced
66,9,0.448489,0.140122,10.0,linear,scale,balanced
69,9,0.448489,0.140122,10.0,linear,auto,balanced


## 3. Дерево решений
1) Попробуйте GridSearchCVразные параметры max_depth(от 1до 49), class_weight( balanced, None) и criterion( entropyи gini) и найдите наилучшую их комбинацию с точки зрения точности. Используйте random_state=21.
2) Создайте датафрейм из результатов поиска по сетке и отсортируйте его по возрастанию по значению rank_test_score, проверьте, есть ли существенная разница между различными комбинациями (иногда более простая модель может дать сопоставимый результат).



In [7]:
tree = DecisionTreeClassifier(random_state=21)
param_grid_tree = {'max_depth': [i for i in range(1, 49+1)],
                   'class_weight': [None, 'balanced'],
                   'criterion': ['gini', 'entropy'],}

gridTree = GridSearchCV(tree, param_grid_tree)
gridTree.fit(X,y)
print("Лучшие параметры:", gridTree.best_params_)
print("Лучшая точность:", gridTree.best_score_)

Лучшие параметры: {'class_weight': None, 'criterion': 'gini', 'max_depth': 16}
Лучшая точность: 0.4816848980738504


In [8]:
results_df_tree = pd.DataFrame(gridTree.cv_results_)

columns_to_show = ['rank_test_score', 'mean_test_score', 'std_test_score',
                   'param_max_depth', 'param_criterion', 'param_class_weight']

results_sorted_tree = results_df_tree[columns_to_show].sort_values('rank_test_score')

results_sorted_tree.head(15)


,rank_test_score,mean_test_score,std_test_score,param_max_depth,param_criterion,param_class_weight
15,1,0.481685,0.091695,16,gini,None
14,2,0.481680,0.087836,15,gini,None
57,3,0.481116,0.123746,9,entropy,None
12,4,0.479894,0.073487,13,gini,None
19,5,0.473378,0.090715,20,gini,None
18,6,0.472783,0.089340,19,gini,None
13,7,0.472772,0.077323,14,gini,None
24,8,0.472200,0.094682,25,gini,None
110,9,0.469211,0.144567,13,gini,balanced
58,10,0.466295,0.122541,10,entropy,None


## 4. Случайный лес
1) Попробуйте GridSearchCVразные параметры n_estimators( 5, 10, 50, 100), max_depth(от 1до 49), class_weight( balanced, None) и criterion( entropyи gini) и найдите наилучшую их комбинацию с точки зрения точности. Используйте random_state=21.
2) Создайте датафрейм из результатов поиска по сетке и отсортируйте его по возрастанию по значению rank_test_score, проверьте, есть ли существенная разница между различными комбинациями (иногда более простая модель может дать сопоставимый результат).

In [16]:
rndForest = RandomForestClassifier(random_state=21)
param_grid_forest = {'max_depth': [i for i in range(1, 49+1)],
                   'class_weight': [None, 'balanced'],
                   'criterion': ['gini', 'entropy'],
                   'n_estimators': [5,10,50,100]}

gridForest = GridSearchCV(rndForest, param_grid_forest)
gridForest.fit(X,y)
print("Лучшие параметры:", gridForest.best_params_)
print("Лучшая точность:", gridForest.best_score_ )

Лучшие параметры: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 14, 'n_estimators': 100}
Лучшая точность: 0.5582164240689692


In [10]:
results_df_forest = pd.DataFrame(gridForest.cv_results_)

columns_to_show = ['rank_test_score', 'mean_test_score', 'std_test_score',
                   'param_max_depth', 'param_criterion','param_n_estimators',  'param_class_weight']

results_sorted_forest = results_df_forest[columns_to_show].sort_values('rank_test_score')

results_sorted_forest.head(15)



,rank_test_score,mean_test_score,std_test_score,param_max_depth,param_criterion,param_n_estimators,param_class_weight
251,1,0.558216,0.151344,14,entropy,100,None
250,2,0.547550,0.169831,14,entropy,50,None
526,3,0.544591,0.154413,34,gini,50,balanced
263,4,0.544586,0.158737,17,entropy,100,None
531,5,0.544582,0.152034,35,gini,100,balanced
527,6,0.543991,0.154848,34,gini,100,balanced
249,7,0.543980,0.172316,14,entropy,10,None
517,8,0.543424,0.159301,32,gini,10,balanced
630,9,0.543394,0.175302,11,entropy,50,balanced
557,10,0.542237,0.167575,42,gini,10,balanced


## 5. Индикатор выполнения
Поиск по сетке может занять довольно много времени, и вы можете задаться вопросом, когда же он закончится.

1) Создайте вручную поиск по сетке для тех же значений параметров случайного леса, перебирая список возможных значений и вычисляя значение cross_val_score для каждой комбинации. Попробуйте увеличить значение n_jobs. Значение равно 5.cvcross_val_score
2) Отслеживайте прогресс с помощью библиотеки tqdm.notebook.
3) Создайте DataFrame из результатов поиска по сетке, в котором столбцы будут соответствовать названиям параметров mean_accuracyи std_accuracy.
4) Отсортируйте по убыванию mean_accuracy, проверьте, есть ли существенная разница между различными комбинациями (иногда более простая модель может дать сопоставимый результат).

In [15]:

results = []  

param_combinations = [
    (depth, weight, crit, n_est)
    for depth in param_grid_forest['max_depth']
    for weight in param_grid_forest['class_weight']
    for crit in param_grid_forest['criterion']
    for n_est in param_grid_forest['n_estimators']
]

for depth, weight, crit, n_est in tqdm(param_combinations, desc="Grid Search"):
    rf = RandomForestClassifier(
        max_depth=depth,
        class_weight=weight,
        criterion=crit,
        n_estimators=n_est,
        n_jobs=-1, 
        random_state=21
    )

    scores = cross_val_score(rf, X, y, cv=5, n_jobs=-1)
    mean_score = np.mean(scores)
    std_score = np.std(scores)
    
    results.append({
        'max_depth': depth,
        'class_weight': weight,
        'criterion': crit,
        'n_estimators': n_est,
        'mean_accuracy': mean_score,
        'std_accuracy': std_score
    })

df_results = pd.DataFrame(results)

df_results = df_results.sort_values(by='mean_accuracy', ascending=False).reset_index(drop=True)

df_results.head(15)

Grid Search:   0%|          | 0/784 [00:00<?, ?it/s]

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for 

,max_depth,class_weight,criterion,n_estimators,mean_accuracy,std_accuracy
0,14,None,entropy,100,0.558216,0.151344
1,14,None,entropy,50,0.547550,0.169831
2,34,balanced,gini,50,0.544591,0.154413
3,17,None,entropy,100,0.544586,0.158737
4,35,balanced,gini,100,0.544582,0.152034
5,34,balanced,gini,100,0.543991,0.154848
6,14,None,entropy,10,0.543980,0.172316
7,32,balanced,gini,10,0.543424,0.159301
8,11,balanced,entropy,50,0.543394,0.175302
9,45,balanced,gini,10,0.542237,0.167575


## 6. Прогнозы
1) Выберите лучшую модель и используйте её для прогнозирования на тестовом наборе данных.
2) Рассчитайте окончательную точность.

In [26]:
rndForest_best = RandomForestClassifier(random_state=21, class_weight=None,
                                        criterion='entropy', n_estimators=100,
                                        n_jobs=-1, max_depth=14)
rndForest_best.fit(X_train,y_train)

y_pred = rndForest_best.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(acc)

0.908284023668639
